# EMA Causal Discovery Analysis

This notebook demonstrates a complete causal discovery workflow using ecological momentary assessment (EMA) data — daily measures of alcohol use, sleep, and mood.

It mirrors the `fastcda_demo_short` workflow but uses the new **fastcausal** API.

**Topics covered:**
1. Load and inspect EMA data
2. Add lag columns and standardize
3. Build temporal prior knowledge
4. Run single causal search (GFCI)
5. Visualize with node styling
6. Compare graphs across parameters
7. Stability (bootstrap) analysis
8. Inspect SEM results

In [ ]:
from fastcausal import FastCausal

fc = FastCausal()

---
## 1. Load data

The bundled `"boston"` dataset contains daily EMA measurements for one participant:

| Variable | Description |
|----------|-------------|
| `alcohol_bev` | Number of alcoholic beverages |
| `TIB` | Time in bed (hours) |
| `TST` | Total sleep time (hours) |
| `PANAS_PA` | Positive affect score |
| `PANAS_NA` | Negative affect score |
| `worry_scale` | Worry severity |
| `PHQ9` | Depression symptom score |

In [ ]:
df = fc.load_sample("boston")
print(f"Shape: {df.shape}")
df.describe()

---
## 2. Add lag columns and standardize

For time-series causal discovery we create lagged versions of each variable (yesterday's values, suffixed `_lag`). These represent potential causes of today's values. We then standardize all columns.

In [ ]:
# Save the original column names for knowledge creation
cols = df.columns.tolist()

# Add lagged columns (n_lags=1 by default)
df_lag = fc.add_lag_columns(df)
print(f"Columns after lagging: {df_lag.columns.tolist()}")

In [ ]:
# Standardize to zero mean, unit variance
df_prep = fc.standardize(df_lag)
df_prep.head()

---
## 3. Temporal prior knowledge

We encode the temporal ordering: lag variables (tier 0) can only be parents of current-day variables (tier 1), never the other way around.

In [ ]:
# create_lag_knowledge automatically separates *_lag columns into tier 0
# and current-day columns into tier 1
knowledge = fc.create_lag_knowledge(cols)
knowledge

---
## 4. Single causal search

Run GFCI with `penalty_discount=1.0`. The algorithm will automatically fit a SEM to quantify edge strengths.

In [ ]:
result, graph = fc.run_search(
    df_prep,
    algorithm="gfci",
    alpha=0.01,
    penalty_discount=1.0,
    knowledge=knowledge,
    run_sem=True,
)

print(f"Nodes: {result['num_nodes']}")
print(f"Edges: {result['num_edges']}")
print("\nEdge list:")
for e in result["edges"]:
    print(f"  {e}")

In [ ]:
# Show all edges (directed and undirected)
fc.show_graph(graph)

In [ ]:
# Directed edges only (cleaner view of causal claims)
fc.show_graph(graph, directed_only=True)

---
## 5. Node styling

Use `fnmatch`-style patterns to highlight variable groups. Rules are applied in order (later rules override earlier ones for the same node).

In [ ]:
node_styles = [
    # Lag variables: dashed border
    {"pattern": "*_lag",        "style": "dotted"},
    # Positive affect: green
    {"pattern": "PANAS_PA*",    "style": "filled", "fillcolor": "lightgreen"},
    # Negative affect: pink
    {"pattern": "PANAS_NA*",    "style": "filled", "fillcolor": "lightpink"},
    # Lag versions of PANAS: combine dotted + color
    {"pattern": "PANAS_PA_lag", "style": "filled,dotted", "fillcolor": "lightgreen"},
    {"pattern": "PANAS_NA_lag", "style": "filled,dotted", "fillcolor": "lightpink"},
    # Alcohol: rectangular, purple
    {"pattern": "alcohol_bev*", "shape": "box", "style": "filled",
     "fillcolor": "purple", "fontcolor": "white"},
]

fc.show_graph(graph, node_styles=node_styles)

In [ ]:
# Styled + directed only
fc.show_graph(graph, node_styles=node_styles, directed_only=True)

---
## 6. Multi-graph comparison

Run GFCI with three different `penalty_discount` values and compare side-by-side. Nodes are pinned at shared positions so changes between graphs are easy to spot.

In [ ]:
result2, graph2 = fc.run_search(
    df_prep, algorithm="gfci", alpha=0.01,
    penalty_discount=2.0, knowledge=knowledge,
)

result3, graph3 = fc.run_search(
    df_prep, algorithm="gfci", alpha=0.01,
    penalty_discount=3.0, knowledge=knowledge,
)

print(f"PD=1: {result['num_edges']} edges")
print(f"PD=2: {result2['num_edges']} edges")
print(f"PD=3: {result3['num_edges']} edges")

In [ ]:
# Compare all edges side-by-side with shared layout
fc.show_n_graphs(
    [graph, graph2, graph3],
    node_styles=node_styles,
    labels=["PD=1.0", "PD=2.0", "PD=3.0"],
    gray_disconnected=True,
    graph_size="10,8",
)

In [ ]:
# Directed edges only
fc.show_n_graphs(
    [graph, graph2, graph3],
    node_styles=node_styles,
    labels=["PD=1.0", "PD=2.0", "PD=3.0"],
    gray_disconnected=True,
    directed_only=True,
    graph_size="10,8",
)

In [ ]:
# Save graphs to PNG files
fc.save_n_graphs(
    [graph, graph2, graph3],
    ["graph_pd1", "graph_pd2", "graph_pd3"],
    node_styles=node_styles,
    labels=["PD=1.0", "PD=2.0", "PD=3.0"],
    gray_disconnected=True,
    graph_size="10,8",
    res=300,
)
print("Saved: graph_pd1.png, graph_pd2.png, graph_pd3.png")

---
## 7. Stability (bootstrap) analysis

Bootstrapped stability analysis runs the algorithm many times on random subsamples and keeps only edges that appear consistently (>= `min_fraction` of runs). This removes spurious edges and gives more reproducible results.

In [ ]:
stab_result, stab_graph = fc.run_stability(
    df_prep,
    algorithm="gfci",
    runs=50,           # increase to 100+ for production use
    min_fraction=0.5,  # edge must appear in >= 50% of runs
    subsample_fraction=0.9,
    alpha=0.01,
    penalty_discount=1.0,
    knowledge=knowledge,
    run_sem=True,
)

print(f"Stable edges ({stab_result['runs']} runs, min_fraction={stab_result['min_fraction']}):")
for e in stab_result["edges"]:
    frac = stab_result["sorted_edge_counts"].get(e, 0)
    print(f"  {e}  ({frac:.0%})")

In [ ]:
# View all edge fractions (including edges below threshold)
import pandas as pd

edge_df = pd.DataFrame(
    [(e, f) for e, f in stab_result["sorted_edge_counts"].items()],
    columns=["edge", "fraction"]
).sort_values("fraction", ascending=False)
edge_df

In [ ]:
if stab_graph is not None:
    fc.show_graph(stab_graph, node_styles=node_styles)
else:
    print("No stable edges found at this threshold.")

---
## 8. SEM results

When `run_sem=True`, fastcausal converts the edge list to a lavaan-style model string and fits it with `semopy`. The estimates are used to color edges in the graph (red = positive, blue = negative).

In [ ]:
if result.get("sem_results"):
    print("=== SEM estimates (single search) ===")
    print(result["sem_results"]["estimates"].to_string())

In [ ]:
if stab_result.get("sem_results"):
    print("=== SEM estimates (stability search) ===")
    print(stab_result["sem_results"]["estimates"].to_string())

---
## Summary

The complete fastcausal EMA workflow:

```
fc.load_sample()  →  fc.add_lag_columns()  →  fc.standardize()
    ↓
fc.create_lag_knowledge()
    ↓
fc.run_search() or fc.run_stability()
    ↓
fc.show_graph() / fc.show_n_graphs() / fc.save_graph()
```

**Next:** See `batch_project.ipynb` for config-driven batch analysis across many participants.